# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliakhtar1010/search-ranking-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [18]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/aliakhtar1010/search-ranking-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [19]:
# Create the evaluation label.
# Used ONLY to evaluate our baseline rule — never as an input to the score.

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Declining pages:", df["is_declining_label"].sum())
print("Base decline rate:", round(df["is_declining_label"].mean(), 3))

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

Rows: 30000
Declining pages: 16262
Base decline rate: 0.542

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [20]:
# SIGNAL 1 — STALENESS

signal1 = df[
    df["days_since_last_update"].notna()
].copy()

signal1["staleness_bucket"] = pd.cut(
    signal1["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=[
        "0-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

staleness_table = (
    signal1
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

staleness_table["declining_rate_pct"] = (
    staleness_table["declining_rate"] * 100
).round(1)

display(
    staleness_table[
        ["staleness_bucket", "n", "declining_rate_pct"]
    ]
)

,staleness_bucket,n,declining_rate_pct
0,0-90 days,20655,51.2
1,91-180 days,9171,61.1
2,181-365 days,169,46.7
3,365+ days,5,60.0


In [21]:
# SIGNAL 2 — SEARCH VISIBILITY (90-DAY IMPRESSIONS)

signal2 = df[
    df["impressions_90d"].notna()
].copy()

signal2["impression_bucket"] = pd.qcut(
    signal2["impressions_90d"],
    q=4,
    duplicates="drop"
)

impression_table = (
    signal2
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        median_impressions=("impressions_90d", "median"),
        declining_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

impression_table["declining_rate_pct"] = (
    impression_table["declining_rate"] * 100
).round(1)

display(
    impression_table[
        [
            "impression_bucket",
            "n",
            "median_impressions",
            "declining_rate_pct"
        ]
    ]
)

,impression_bucket,n,median_impressions,declining_rate_pct
0,"(0.999, 81.0]",7503,10.0,37.6
1,"(81.0, 731.0]",7499,300.0,60.5
2,"(731.0, 3615.25]",7498,1616.0,62.6
3,"(3615.25, 517715.0]",7500,9579.5,56.2


Rule idea

I want the baseline to prioritize pages that are no longer recently updated but still have meaningful search visibility. The rule should remain simple enough for a content manager to understand without machine learning.

Signal 1 — Staleness: MIXED

Pages updated 91–180 days ago had a 61.1% observed decline rate compared with 51.2% for pages updated within 90 days. However, the relationship was not consistently stronger for older buckets, and the 181+ day groups had very small sample sizes. Staleness therefore appears directionally useful, but the data does not support a simple “older is always worse” claim.

Signal 2 — Search visibility: MIXED

The lowest-impression bucket had a 37.6% observed decline rate, while the next three buckets were 60.5%, 62.6%, and 56.2%. Pages with meaningful existing visibility therefore showed more decline than very low-visibility pages, although the relationship was not strictly increasing with impressions.

Baseline rule

A page is prioritized for refresh review if it has not been updated for at least 90 days and has at least 731 impressions over the trailing 90-day window. Qualifying pages are ranked by impressions so that pages with more existing search visibility are reviewed first.

Reason code: stale_but_visible

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
import os
import numpy as np

# Freeze the baseline thresholds
STALE_THRESHOLD = 90
VISIBILITY_THRESHOLD = 731

# Make a working copy
baseline = df.copy()

# Conditions used by the human-readable rule
baseline["is_stale"] = (
    baseline["days_since_last_update"] >= STALE_THRESHOLD
)

baseline["is_visible"] = (
    baseline["impressions_90d"] >= VISIBILITY_THRESHOLD
)

# Score:
# Only stale + visible pages receive a non-zero score.
# Among those pages, more impressions = higher review priority.
baseline["action_score"] = np.where(
    baseline["is_stale"] & baseline["is_visible"],
    baseline["impressions_90d"],
    0
)

# One reason code and one action
baseline["reason_code"] = np.where(
    baseline["action_score"] > 0,
    "stale_but_visible",
    "not_prioritized"
)

baseline["action_label"] = np.where(
    baseline["action_score"] > 0,
    "review_for_refresh",
    "no_action"
)

# Rank highest-priority pages first
baseline = baseline.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

# Label ONLY for evaluation.
# trend_direction is NOT used to calculate the score.
baseline["declining_label"] = (
    baseline["trend_direction"] == "down"
).astype(int)

print("Total pages:", len(baseline))
print("Pages prioritized:", (baseline["action_score"] > 0).sum())
print(
    "Prioritized %:",
    round((baseline["action_score"] > 0).mean() * 100, 2)
)

baseline[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "action_score",
        "reason_code",
        "action_label"
    ]
].head(10)

Total pages: 30000
Pages prioritized: 5992
Prioritized %: 19.97


,content_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label
0,content_5fe46e04994d,104,517715,517715,stale_but_visible,review_for_refresh
1,content_2dba2b1f9536,104,443434,443434,stale_but_visible,review_for_refresh
2,content_2c2606c5d176,104,347399,347399,stale_but_visible,review_for_refresh
3,content_cb112fce36be,104,309910,309910,stale_but_visible,review_for_refresh
4,content_9532f197bbc8,104,309192,309192,stale_but_visible,review_for_refresh
5,content_36ff89c8214e,104,295097,295097,stale_but_visible,review_for_refresh
6,content_b28d1efd668f,104,286608,286608,stale_but_visible,review_for_refresh
7,content_813e88069237,104,233561,233561,stale_but_visible,review_for_refresh
8,content_c21024970297,104,211366,211366,stale_but_visible,review_for_refresh
9,content_c8e9d6ab9013,104,208678,208678,stale_but_visible,review_for_refresh


In [23]:
def precision_at_k(data, k):
    top_k = data.head(k)
    return top_k["declining_label"].mean()

base_rate = baseline["declining_label"].mean()

print("Base decline rate:", round(base_rate, 3))
print("Precision@10:", round(precision_at_k(baseline, 10), 3))
print("Precision@20:", round(precision_at_k(baseline, 20), 3))
print("Precision@50:", round(precision_at_k(baseline, 50), 3))

Base decline rate: 0.542
Precision@10: 0.6
Precision@20: 0.45
Precision@50: 0.44


In [24]:
os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "content_id",
    "client_id",
    "action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d"
]

baseline[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved: work/outputs/baseline_action_score.csv")

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10_review = baseline.head(10)[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position",
        "action_score",
        "reason_code",
        "action_label"
    ]
].copy()

display(top10_review)

,content_id,days_since_last_update,impressions_90d,ctr,avg_position,action_score,reason_code,action_label
0,content_5fe46e04994d,104,517715,0.14,4.2,517715,stale_but_visible,review_for_refresh
1,content_2dba2b1f9536,104,443434,0.21,27.9,443434,stale_but_visible,review_for_refresh
2,content_2c2606c5d176,104,347399,0.53,4.2,347399,stale_but_visible,review_for_refresh
3,content_cb112fce36be,104,309910,0.16,5.6,309910,stale_but_visible,review_for_refresh
4,content_9532f197bbc8,104,309192,0.87,2.0,309192,stale_but_visible,review_for_refresh
5,content_36ff89c8214e,104,295097,0.05,7.3,295097,stale_but_visible,review_for_refresh
6,content_b28d1efd668f,104,286608,0.06,26.2,286608,stale_but_visible,review_for_refresh
7,content_813e88069237,104,233561,0.06,26.2,233561,stale_but_visible,review_for_refresh
8,content_c21024970297,104,211366,0.41,5.1,211366,stale_but_visible,review_for_refresh
9,content_c8e9d6ab9013,104,208678,0.00,9.7,208678,stale_but_visible,review_for_refresh


### Top-10 Review

1. `content_5fe46e04994d` — **Action:** review for refresh. **Why:** 104 days since update and 517,715 impressions make it the highest-visibility stale page in the queue. **What would make it wrong:** the traffic change may be caused by seasonality, search-demand changes, or external ranking factors rather than stale content.

2. `content_2dba2b1f9536` — **Action:** review for refresh. **Why:** 104 days since update and 443,434 impressions create a large visible opportunity. **What would make it wrong:** high visibility alone does not prove that refreshing the content will improve performance.

3. `content_2c2606c5d176` — **Action:** review for refresh. **Why:** it passes both the staleness and visibility thresholds and has 347,399 impressions. **What would make it wrong:** the page may already satisfy search intent despite being older.

4. `content_cb112fce36be` — **Action:** review for refresh. **Why:** 104 days since update with 309,910 impressions places it high in the transparent baseline ranking. **What would make it wrong:** declining search demand rather than content quality may explain weak future performance.

5. `content_9532f197bbc8` — **Action:** review for refresh. **Why:** it is stale by the baseline definition and still receives 309,192 impressions. **What would make it wrong:** the page may not require substantive changes even if it is old.

6. `content_36ff89c8214e` — **Action:** review for refresh. **Why:** 104 days since update and 295,097 impressions indicate a visible stale page. **What would make it wrong:** external SERP competition may be responsible for performance changes.

7. `content_b28d1efd668f` — **Action:** review for refresh. **Why:** the page meets both baseline conditions and has 286,608 impressions. **What would make it wrong:** a refresh may have little effect if the underlying topic is losing demand.

8. `content_813e88069237` — **Action:** review for refresh. **Why:** 104 days since update and 233,561 impressions keep it high in the queue. **What would make it wrong:** the page may remain accurate and useful despite not being recently updated.

9. `content_c21024970297` — **Action:** review for refresh. **Why:** it is beyond the 90-day threshold and still has 211,366 impressions. **What would make it wrong:** impressions measure visibility, not whether the content itself is deficient.

10. `content_c8e9d6ab9013` — **Action:** review for refresh. **Why:** 104 days since update and 208,678 impressions make it a meaningful search asset to inspect. **What would make it wrong:** the observed issue may come from ranking or demand changes that a content refresh cannot fix.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

The baseline performs slightly above the overall decline base rate at the top 10: Precision@10 is 0.60 compared with a base decline rate of 0.542. However, Precision@20 falls to 0.45 and Precision@50 to 0.44. This suggests the rule concentrates some useful cases at the very top but becomes weak as the queue expands.

The top-ranked pages also all have the same days_since_last_update value of 104 days. This means impressions are doing most of the ordering within the top of the queue, while staleness mainly acts as an eligibility threshold. That is a limitation of this simple baseline and gives the Week-5 model an opportunity to learn more useful combinations of signals.

Leakage check

The baseline score uses only days_since_last_update and impressions_90d. Neither trend_direction, trend_pct, nor is_declining_label is used to calculate the score. The decline label is used only after ranking to evaluate Precision@K. No future-window or label-derived information is used as an input to the baseline.

Baseline verdict

This is a deliberately simple and interpretable baseline. It is useful enough to establish a benchmark, but its declining-page concentration is only modest at the very top and weaker beyond the top 10. The Week-5 model should be compared against this frozen rule on the same data and metric.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

forbidden_inputs = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "declining_label"
]

print("Baseline score inputs:")
for col in baseline_inputs:
    print("SAFE:", col)

print("\nLabel / outcome fields excluded from score:")
for col in forbidden_inputs:
    print("EXCLUDED:", col)

Baseline score inputs:
SAFE: days_since_last_update
SAFE: impressions_90d

Label / outcome fields excluded from score:
EXCLUDED: trend_direction
EXCLUDED: trend_pct
EXCLUDED: is_declining_label
EXCLUDED: declining_label


In [27]:
import json

metrics = {
    "base_decline_rate": float(base_rate),
    "precision_at_10": float(precision_at_k(baseline, 10)),
    "precision_at_20": float(precision_at_k(baseline, 20)),
    "precision_at_50": float(precision_at_k(baseline, 50)),
    "stale_threshold_days": int(STALE_THRESHOLD),
    "visibility_threshold_impressions": int(VISIBILITY_THRESHOLD),
    "pages_prioritized": int((baseline["action_score"] > 0).sum())
}

os.makedirs("work/outputs", exist_ok=True)

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(metrics)

{'base_decline_rate': 0.5420666666666667, 'precision_at_10': 0.6, 'precision_at_20': 0.45, 'precision_at_50': 0.44, 'stale_threshold_days': 90, 'visibility_threshold_impressions': 731, 'pages_prioritized': 5992}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.